In [42]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [43]:
df = pd.read_csv("clean_dataset.csv")

In [44]:
X = df.drop("buy_price", axis=1)

In [45]:
y = df["buy_price"]

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [47]:
print("Taille du train :", X_train.shape)
print("Taille du test :", X_test.shape)

Taille du train : (17393, 14)
Taille du test : (4349, 14)


In [48]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

In [49]:
lr = LinearRegression()

In [50]:
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
print("Linear Regression RMSE :", lr_rmse)

Linear Regression RMSE : 287159.7197360807


In [51]:
rf = RandomForestRegressor(random_state=42)

rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

In [53]:
rf_grid = GridSearchCV(
    rf, rf_params, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1
)
rf_grid.fit(X_train, y_train)
rf_pred = rf_grid.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
print("Random Forest RMSE :", rf_rmse)
print("Best params RF:", rf_grid.best_params_)

Random Forest RMSE : 21474.903912490438
Best params RF: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


In [52]:
xgbr = xgb.XGBRegressor(random_state=42, objective="reg:squarederror")

xgb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1],
    "max_depth": [3, 5]
}

In [54]:
xgb_grid = GridSearchCV(
    xgbr, xgb_params, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
xgb_pred = xgb_grid.predict(X_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
print("XGBoost RMSE :", xgb_rmse)
print("Best params XGB:", xgb_grid.best_params_)

XGBoost RMSE : 36042.00608301163
Best params XGB: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}


In [55]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "RMSE": [lr_rmse, rf_rmse, xgb_rmse]
}).sort_values("RMSE")

print("\n📊 Comparaison des modèles :")
print(results)


📊 Comparaison des modèles :
               Model           RMSE
1      Random Forest   21474.903912
2            XGBoost   36042.006083
0  Linear Regression  287159.719736
